In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd #to store data as csv or excel format
import time
from datetime import date

WRITE_FILE_PATH = r'c:\Interest_rate_files'
CHROME_DRIVER_PATH = 'c:\\Interest_rate_files\\chromedriver-win64\\chromedriver.exe'
URL = "https://www.ucobank.com/interest-rates-on-deposit-schemes" 

date1 = (date.today()).strftime('%d%m%Y') #Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' #Set directory path to write file

# Setup Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Provide the path to your ChromeDriver
service = Service(CHROME_DRIVER_PATH)

# Initialize the WebDriver
driver = webdriver.Chrome(executable_path=CHROME_DRIVER_PATH)

try:
    # Open the webpage
    driver.get(URL)

    # Wait until the content is loaded
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    # Give it extra time if needed
    time.sleep(5)

    # Get the page source after rendering JavaScript
    page_content = driver.page_source

    # You can parse this with BeautifulSoup if needed
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(page_content, 'html.parser')
    table=soup.find_all("table")
    table1 = table[1]
    rows=table1.find_all("tr")
    
    print("Today's date =",date1)
    print("Interest rate file name = ",file_name)
    
    list_of_interest_rates = []
    tax_saver_row = []
    row_count = 0
    for i in rows[1:]:  #Skip heading
        data = i.find_all("td")
        #print(data)
        row=['UCO Bank']
        row.extend([tr.text for tr in data[:2]])
        if row_count < 11: row.extend(['',str(float(row[2][:3]) + 0.25) + '%','']) 
        else: row.extend(['',str(float(row[2][:3]) + 0.5) + '%','',])
        if row_count == 16:
            tax_saver_row = ['UCO Bank','Tax-saver Fixed Deposit',row[2],'',row[4],'']
            list_of_interest_rates.append(tax_saver_row)
        list_of_interest_rates.append(row)
        row_count += 1
    list_of_interest_rates[-1][1] = 'Above 5 years'
    #print(list_of_interest_rates)
    
    pd.set_option('display.max_columns', None)    
    int_rate_df=pd.DataFrame(list_of_interest_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                            'sr_rate','annualised_sr_rate'])
    print(int_rate_df)
    int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
    print("\nUCO Bank = Success. Number of rows added = ", len(list_of_interest_rates))
finally:
    # Close the driver
    driver.quit()

Today's date = 19112024
Interest rate file name =  c:\Interest_rate_files\Interest_rate_v2_19112024.csv
   bank_name                   tenure gen_rate annualised_gen_rate sr_rate  \
0   UCO Bank                7-14 days    2.90%                       3.15%   
1   UCO Bank               15-29 days    2.90%                       3.15%   
2   UCO Bank               30-45 days    3.00%                       3.25%   
3   UCO Bank               46-60 days    3.50%                       3.75%   
4   UCO Bank               61-90 days    3.50%                       3.75%   
5   UCO Bank              91-120 days    4.50%                       4.75%   
6   UCO Bank             121-150 days    4.50%                       4.75%   
7   UCO Bank             151-180 days    5.00%                       5.25%   
8   UCO Bank             181-332 days    5.50%                       5.75%   
9   UCO Bank                 333 days    7.30%                       7.55%   
10  UCO Bank                  334-364 